# Setting up a proper undersampling pipeline

Code repository for the book:

[Imbalanced Data: Myths, Mistakes and Modern Solutions](https://www.trainindata.com/p/imbalanced-data-myths-mistakes-solutions-book)

When we undersample, we must resample **only the training data** and leave the test data at its original class distribution. That is because we want to evaluate the performance of the model using the real, original distribution.

`imbalanced-learn` provides its own `Pipeline` that solves this: it applies the undersampler during `fit`, and leaves the data untouched when predicting. 

In this notebook we build such a pipeline to train and evaluate XGBoost on the Bank Marketing dataset with cross-validation.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="Could not infer format")

In [2]:
from feature_engine.encoding import OrdinalEncoder

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

from xgboost import XGBClassifier

## Load the data

The Bank Marketing dataset records a phone marketing campaign. The target indicates whether a client subscribed to a term deposit, which is the minority class.

In [3]:
data = fetch_openml(name="bank-marketing", version=1, as_frame=True, parser="auto")

X = data.data.copy()
y = (data.target == "2").astype(int)  # 1 = subscribed, 0 = did not subscribe

print(f"Observations: {X.shape[0]}, features: {X.shape[1]}")
y.value_counts(normalize=True)

Observations: 45211, features: 16


Class
0    0.883015
1    0.116985
Name: proportion, dtype: float64

## Split into train and test

We separate a held-out test set, keeping the class proportions in both sets with `stratify`. The test set stays at its original distribution and is never undersampled.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"Imbalance ratio (majority:minority): {imbalance_ratio:.1f} : 1")

Train: 31647 rows | Test: 13564 rows
Imbalance ratio (majority:minority): 7.5 : 1


## Build the undersampling pipeline

We use imbalanced-learn's `Pipeline`, **not** scikit-learn's. The difference matters: imbalanced-learn's pipeline calls `fit_resample` on the undersampler during training, but skips it when predicting. So the undersampling is applied to the training data only, and the data we evaluate on keeps its original class distribution.

`RandomUnderSampler` with the default `sampling_strategy="auto"` removes majority-class observations at random until the classes are balanced 1:1.

In [5]:
pipeline = Pipeline(
    steps=[
        ("encoder", OrdinalEncoder(encoding_method="arbitrary")),
        ("undersampler", RandomUnderSampler(random_state=0)),
        ("model", XGBClassifier(random_state=0, n_jobs=-1, eval_metric="logloss")),
    ]
)

pipeline

,steps,"[('encoder', ...), ('undersampler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,encoding_method,'arbitrary'
,variables,None
,missing_values,'raise'
,ignore_format,False
,unseen,'ignore'
,sampling_strategy,'auto'
,random_state,0


## Evaluate with cross-validation

Because the undersampler lives inside the pipeline, it runs separately on each training fold, while every validation fold keeps the original class distribution. This gives a leak-free estimate of performance. We report ROC-AUC and PR-AUC (average precision), both threshold-independent metrics, with their dispersion across folds.

In [6]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=10)

scores = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=["roc_auc", "average_precision"],
    n_jobs=-1,
)

for metric in ["test_roc_auc", "test_average_precision"]:
    print(f"{metric:>25}: {scores[metric].mean():.4f} +/- {scores[metric].std():.4f}")

             test_roc_auc: 0.9233 +/- 0.0031
   test_average_precision: 0.5732 +/- 0.0072


## Fit and evaluate on the held-out test set

Finally, we fit the whole pipeline on the training set and evaluate on the test set. The undersampler resamples the training data, the model is trained on the balanced set, and the test set is scored at its original distribution.

In [7]:
pipeline.fit(X_train, y_train)

y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"Test ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"Test PR-AUC : {average_precision_score(y_test, y_proba):.4f}")

Test ROC-AUC: 0.9225
Test PR-AUC : 0.5721


## A note on efficiency

This pipeline is correct, but notice that it re-runs the undersampler every time the model is fitted: once per cross-validation fold, and again on each candidate during a hyperparameter search. Random undersampling is cheap, but the cleaning methods are not, and repeating them on every fit is wasteful when we want to compare several models on the same undersampled data. In the next notebook we undersample once and train all the models on top of it.